# Experiments: seed averaging, lower learning rate, recency weighting

What is tested, all under the unchanged baseline protocol (train 2015-2021,
train_tail early stopping, full 2022 val, same 193 features):

1. **Seed averaging**: 5 XGBoost seeds (0-4) per generator, ensembled by
   mean probability and by mean within-generator percentile rank.
2. **Lower learning rate, more trees**: lr 0.03 with 1000 and 1500 trees
   (baseline: lr 0.05, 400 trees).
3. **Recency weighting**: per-case training weights decaying exponentially
   with case-start year (decay 0.9 and 0.75 per year before 2022; 2021
   weight 1.0), motivated by the measured year drift in ON-rates.

Comparison target: baseline micro AUC 0.9802 (seed 0, same protocol). This
is a cheap screen on the val-2022 split; any variant that beats 0.9802
beyond noise gets confirmed on the pooled LOYO OOF (0.9758) before being
adopted. Expected runtime: roughly 45-60 minutes for all cells.

In [1]:
import numpy as np
import pandas as pd
import xgboost as xgb
import yaml
from sklearn.metrics import roc_auc_score

HOURS = 168

In [2]:
DATA = "../data/kernel"
EXT = "../data/extended"

price = pd.read_csv(f"{DATA}/Historical_day_ahead_price_2015_2025.csv")
inflow = pd.read_csv(f"{DATA}/Historical_inflow_1958_2025.csv")
volume = pd.read_csv(f"{DATA}/Historical_volume_2015_2024.csv")
water = pd.read_csv(f"{DATA}/Synthetic_water_value_2015_2024.csv")
target = pd.read_csv(f"{DATA}/Unit_commitment_decisions.csv")
min_flow = pd.read_csv(f"{EXT}/Constraint_min_flow.csv")
min_volume = pd.read_csv(f"{EXT}/Constraint_min_volume.csv")
topology = yaml.safe_load(open(f"{EXT}/Tokke_Vinje_topology.yaml"))


def indexed(df):
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"], utc=True, format="mixed")
    return df.sort_values("date").set_index("date")


price_h = indexed(price)["Day-ahead price (EUR/MWh)"].resample("1h").mean()
inf_wide = indexed(inflow)
vol_wide = indexed(volume)
wv_wide = indexed(water)
mf_wide = indexed(min_flow)
mv_wide = indexed(min_volume)

RESERVOIRS = list(vol_wide.columns)
CAPACITY = {r: float(topology["model"]["reservoir"][r]["max_vol"]) for r in RESERVOIRS}

target["starttime"] = pd.to_datetime(target["starttime"], utc=True, format="mixed")
target = target.sort_values("starttime").reset_index(drop=True)

GENERATORS = sorted(
    {c.removeprefix("result_committed_").rsplit("_t", 1)[0]
     for c in target.columns if c.startswith("result_committed")}
)

connections = topology["connections"]


def supplying_reservoirs(name, kind):
    if kind == "reservoir":
        return {name}
    out = set()
    for e in connections:
        if e["to"] == name and e["to_type"] == kind and e["from_type"] in {"reservoir", "tunnel"}:
            out |= supplying_reservoirs(e["from"], e["from_type"])
    return out


FEEDERS = {}
for gen in GENERATORS:
    plant = next(e["to"] for e in connections
                 if e["from"] == gen and e["from_type"] == "generator" and e["to_type"] == "plant")
    FEEDERS[gen] = sorted(supplying_reservoirs(plant, "plant"))

print("cases:", len(target), " generators:", len(GENERATORS))

cases: 2922  generators: 14


## Feature engineering and split

What is tested: the unchanged 193-column feature set and the unchanged
baseline split (train starts < 2022-01-01, stop on the last 90 days of
train, val = full 2022), copied verbatim from
`notebooks/xgboost_baseline.ipynb`.

In [3]:
def lookup(frame, timestamps, columns=None):
    f = frame if columns is None else frame[columns]
    return f.reindex(pd.DatetimeIndex(timestamps)).to_numpy(dtype=np.float32)


def feature_matrix(cases):
    starts = pd.DatetimeIndex(cases["starttime"])
    n = len(starts)
    ts = pd.DatetimeIndex(np.repeat(starts.to_numpy(), HOURS)) + pd.to_timedelta(
        np.tile(np.arange(HOURS), n), unit="h"
    )
    price = lookup(price_h, ts).reshape(n, HOURS)
    initial = lookup(vol_wide, starts, RESERVOIRS)
    terminal = lookup(wv_wide, starts + pd.Timedelta(hours=HOURS), RESERVOIRS)
    inflow = lookup(inf_wide, ts).reshape(n, HOURS, -1)
    minflow = lookup(mf_wide, ts).reshape(n, HOURS, -1)
    minvol = lookup(mv_wide, ts).reshape(n, HOURS, -1)

    F = {}

    def add(name, v):
        v = np.asarray(v, dtype=np.float32)
        if v.shape == (n,):
            v = np.repeat(v, HOURS)
        elif v.shape == (n, HOURS):
            v = v.reshape(-1)
        F[name] = v

    h = np.tile(np.arange(HOURS, dtype=np.float32), n)
    add("horizon_hour", h)
    add("hours_remaining", HOURS - 1 - h)
    add("hour_of_day", ts.hour.to_numpy())
    add("day_of_week", ts.dayofweek.to_numpy())
    add("month", ts.month.to_numpy())
    add("doy_sin", np.sin(2 * np.pi * ts.dayofyear.to_numpy() / 365.25))
    add("doy_cos", np.cos(2 * np.pi * ts.dayofyear.to_numpy() / 365.25))

    add("price", price)
    for stat, v in (("mean", price.mean(1)), ("std", price.std(1)),
                    ("min", price.min(1)), ("max", price.max(1))):
        add(f"price_week_{stat}", v)
    add("price_minus_week_mean", price - price.mean(1, keepdims=True))
    add("price_week_rank", pd.DataFrame(price).rank(axis=1, pct=True).to_numpy())
    daily = price.reshape(n, 7, 24)
    for stat, v in (("mean", daily.mean(2)), ("min", daily.min(2)), ("max", daily.max(2))):
        add(f"price_day_{stat}", np.repeat(v, 24, axis=1))
    add("price_prefix_mean", price.cumsum(1) / np.arange(1, HOURS + 1))
    add("price_suffix_mean", price[:, ::-1].cumsum(1)[:, ::-1] / np.arange(HOURS, 0, -1))
    for off in (-24, -6, -1, 1, 6, 24):
        add(f"price_offset_{off:+d}", price[:, np.clip(np.arange(HOURS) + off, 0, HOURS - 1)])

    for j, res in enumerate(RESERVOIRS):
        add(f"init_vol__{res}", initial[:, j])
        add(f"init_frac__{res}", initial[:, j] / CAPACITY[res])
        add(f"term_val__{res}", terminal[:, j])
        add(f"price_minus_val__{res}", price - terminal[:, j, None])

    for j, name in enumerate(inf_wide.columns):
        flow = inflow[:, :, j]
        cum = flow.cumsum(1) * (3600 / 1e6)
        add(f"inflow__{name}", flow)
        add(f"inflow_week_mean__{name}", flow.mean(1))
        add(f"inflow_cum_Mm3__{name}", cum)
        add(f"inflow_rem_Mm3__{name}", cum[:, -1:] - cum)

    for j, name in enumerate(mf_wide.columns):
        add(f"minflow__{name}", minflow[:, :, j])
        add(f"minflow_week_max__{name}", minflow[:, :, j].max(1))
    for j, name in enumerate(mv_wide.columns):
        add(f"minvol__{name}", minvol[:, :, j])
        add(f"minvol_week_max__{name}", minvol[:, :, j].max(1))
        add(f"init_margin_to_minvol__{name}", initial[:, RESERVOIRS.index(name), None] - minvol[:, :, j])

    return pd.DataFrame(F, dtype=np.float32)


def generator_features(base, gen):
    X = base.copy()
    feeders = FEEDERS[gen]
    capacity = sum(CAPACITY[r] for r in feeders)
    total_init = sum(base[f"init_vol__{r}"] for r in feeders)
    value = sum(base[f"term_val__{r}"] * CAPACITY[r] for r in feeders) / capacity
    cum = sum(base[f"inflow_cum_Mm3__{r}"] for r in feeders)
    rem = sum(base[f"inflow_rem_Mm3__{r}"] for r in feeders)

    X["local_init_frac"] = total_init / capacity
    X["local_term_val"] = value
    X["local_price_minus_val"] = base["price"] - value
    X["local_inflow"] = sum(base[f"inflow__{r}"] for r in feeders)
    X["local_cum_frac"] = cum / capacity
    X["local_rem_frac"] = rem / capacity
    X["local_available_frac"] = (total_init + cum + rem) / capacity
    return X


def labels_for(cases, gen):
    cols = [f"result_committed_{gen}_t{t}" for t in range(HOURS)]
    return cases[cols].to_numpy(dtype=np.uint8).reshape(-1)


split = pd.Timestamp("2022-01-01", tz="UTC")
train_cases = target[target["starttime"] < split].reset_index(drop=True)
val_cases = target[target["starttime"] >= split].reset_index(drop=True)
stop_cases = train_cases[
    train_cases["starttime"] >= split - pd.Timedelta(days=90)
].reset_index(drop=True)
print("train cases:", len(train_cases), " stop cases:", len(stop_cases),
      " val cases:", len(val_cases))

Xtr_base = feature_matrix(train_cases)
Xst_base = feature_matrix(stop_cases)
Xva_base = feature_matrix(val_cases)
print("feature matrix:", Xtr_base.shape)

train cases: 2557  stop cases: 90  val cases: 365
feature matrix: (429576, 186)


## Fit helper

What is tested: one shared fit loop for all experiments — per-generator
XGBoost with the train_tail protocol, configurable seed, hyperparameters
and per-case training weights (repeated over each case's 168 hours).

In [4]:
def fit_all(seed=0, params=None, case_weights=None):
    p = dict(n_estimators=400, learning_rate=0.05, max_depth=7,
             subsample=0.8, colsample_bytree=0.8, tree_method="hist",
             eval_metric="auc", early_stopping_rounds=25)
    p.update(params or {})
    p["random_state"] = seed

    preds, rows = {}, []
    for gen in GENERATORS:
        Xtr = generator_features(Xtr_base, gen)
        Xst = generator_features(Xst_base, gen)
        Xva = generator_features(Xva_base, gen)
        ytr = labels_for(train_cases, gen)
        yst = labels_for(stop_cases, gen)
        yva = labels_for(val_cases, gen)

        if np.unique(ytr).size == 1:
            preds[gen] = np.full(len(yva), float(ytr[0]), dtype=np.float32)
            rows.append({"generator": gen, "trees": 0, "AUC": None})
            continue

        clf = xgb.XGBClassifier(**p)
        w = None if case_weights is None else np.repeat(case_weights, HOURS)
        clf.fit(Xtr, ytr, sample_weight=w, eval_set=[(Xst, yst)], verbose=False)
        preds[gen] = clf.predict_proba(Xva)[:, 1].astype(np.float32)
        rows.append({"generator": gen, "trees": clf.best_iteration + 1,
                     "AUC": roc_auc_score(yva, preds[gen])})
    return preds, pd.DataFrame(rows).set_index("generator")


def micro(preds):
    y_all = np.concatenate([labels_for(val_cases, g) for g in GENERATORS])
    p_all = np.concatenate([preds[g] for g in GENERATORS])
    return roc_auc_score(y_all, p_all)

## Experiment 1: seed averaging

What is tested: 5 seeds (0-4) per generator under the baseline protocol.
Printed: micro AUC per single seed, then the two ensembles — mean
probability and mean within-generator percentile rank — plus the
per-generator AUC table of the mean-probability ensemble. Comparison:
seed 0 alone is the 0.9802 baseline.

In [5]:
SEEDS = range(5)
seed_preds = {}
for seed in SEEDS:
    seed_preds[seed], _ = fit_all(seed=seed)
    print(f"seed {seed}: micro AUC {micro(seed_preds[seed]):.4f}")

mean_prob = {
    g: np.mean([seed_preds[s][g] for s in SEEDS], axis=0).astype(np.float32)
    for g in GENERATORS
}
mean_rank = {
    g: pd.DataFrame({s: seed_preds[s][g] for s in SEEDS})
        .rank(pct=True).mean(axis=1).to_numpy(dtype=np.float32)
    for g in GENERATORS
}
print(f"ensemble mean prob: micro AUC {micro(mean_prob):.4f}")
print(f"ensemble mean rank: micro AUC {micro(mean_rank):.4f}")

rows = []
for g in GENERATORS:
    rows.append({"generator": g,
                 "AUC": roc_auc_score(labels_for(val_cases, g), mean_prob[g])})
pd.DataFrame(rows).set_index("generator").round(4)

seed 0: micro AUC 0.9802
seed 1: micro AUC 0.9804
seed 2: micro AUC 0.9804
seed 3: micro AUC 0.9794
seed 4: micro AUC 0.9790
ensemble mean prob: micro AUC 0.9802
ensemble mean rank: micro AUC 0.9492


,AUC
generator,
Byrte_G1,0.9954
Haukeli_G1,0.9404
Hogga_G1,0.9730
Kjela_G1,0.9815
Lio_G1,0.9217
Songa_G1,0.9972
Tokke_G1,0.9927
Tokke_G2,0.9840
Tokke_G3,0.9892


## Experiment 2: lower learning rate, more trees

What is tested: lr 0.03 with 1000 trees and with 1500 trees, seed 0,
everything else unchanged. Printed: micro AUC per configuration and the
per-generator tree counts (the train_tail stop set is in-sample, so the
models are expected to run at or near the cap). Comparison: 0.9802.

In [6]:
for n_trees, lr in ((1000, 0.03), (1500, 0.03)):
    preds, trees = fit_all(params={"n_estimators": n_trees, "learning_rate": lr})
    print(f"lr {lr}, {n_trees} trees: micro AUC {micro(preds):.4f}")
    print(trees["trees"].to_string())
    print()

lr 0.03, 1000 trees: micro AUC 0.9803
generator
Byrte_G1          1000
Haukeli_G1        1000
Hogga_G1          1000
Kjela_G1           965
Lio_G1             999
Songa_G1           412
Tokke_G1          1000
Tokke_G2          1000
Tokke_G3          1000
Tokke_G4          1000
Vesle Kjela_G1    1000
Vinje_G1          1000
Vinje_G2           999
Vinje_G3           999

lr 0.03, 1500 trees: micro AUC 0.9804
generator
Byrte_G1          1096
Haukeli_G1        1416
Hogga_G1          1498
Kjela_G1           965
Lio_G1            1314
Songa_G1           412
Tokke_G1          1499
Tokke_G2          1497
Tokke_G3          1495
Tokke_G4          1500
Vesle Kjela_G1    1447
Vinje_G1          1490
Vinje_G2          1499
Vinje_G3          1499



## Experiment 3: recency weighting

What is tested: per-case training weights `decay ** (2021 - year)` for
decay 0.9 (mild: 2015 weight 0.53) and decay 0.75 (strong: 2015 weight
0.13), seed 0, everything else unchanged. The stop set is left
unweighted. Printed: micro AUC per decay. Comparison: 0.9802 (all
weights 1.0).

In [7]:
train_year = train_cases["starttime"].dt.year.to_numpy()
for decay in (0.9, 0.75):
    w = (decay ** (2021 - train_year)).astype(np.float32)
    preds, _ = fit_all(case_weights=w)
    print(f"decay {decay}: micro AUC {micro(preds):.4f}  "
          f"(2015 weight {w.min():.2f}, 2021 weight {w.max():.2f})")

decay 0.9: micro AUC 0.9804  (2015 weight 0.53, 2021 weight 1.00)
decay 0.75: micro AUC 0.9790  (2015 weight 0.18, 2021 weight 1.00)
